# 10. v2 진단 그래프 — 임계값 · 구분력 · 변수 기여

**A. 판정·순위** (oof.csv만 필요): 점수 분포+임계값 / PR 곡선 / 검사 곡선 / 점수 타임라인
**B. 변수 수준** (feature + reference 필요): 편차 히트맵 / 단변량 AUC / 묶음 ablation / logit 계수 안정성

점수 축 `−log10(1−pct)`: 2 = 정상 생산 샷의 1%만 이보다 높음, 3 = 0.1%

In [ ]:
%load_ext autoreload
%autoreload 2
import os, glob
import pandas as pd
from scope import apply_scope
from model_v2 import EV_DEV, EV_DELTA, ST_FREE
from viz_v2 import (plot_score_threshold, plot_pr, plot_inspection, plot_score_timeline,
                    deviation_heatmap, univariate_auc, plot_univariate_auc,
                    group_ablation, plot_group_ablation, logit_coef_stability, plot_coef_stability)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)

V2 = './outputs/v2_noinit'
OUT = './outputs/v2_diag'
os.makedirs(OUT, exist_ok=True)
PRODUCTS = ['cn7', 'rg3']
MAIN = 'E2_U3+logit'                                     # 메인 (사전 고정)
COMPARE = ['E2_U3+logit', 'E1_U3', 'S_rf|sup_free', 'U_knn|free']   # 비교용 최대 4개
REF_FILE = './base/ref_{}_features_2020-10-01_2020-11-30.csv'      # 08에서 쓴 reference 캐시 경로로 맞출 것

OOF = pd.read_csv(f'{V2}/oof.csv')
runs = pd.read_csv('./base/base_runs.csv', encoding='utf-8-sig')   # BOM 제거
OOF['plan_date'] = OOF['run_id'].map(dict(zip(runs['run_id'], runs['생산계획일'])))

FEAT, REF = {}, {}
for p in PRODUCTS:
    FEAT[p] = apply_scope(pd.read_csv(f'./base/{p}_features.csv', parse_dates=['TimeStamp']), True, False)
    FEAT[p]['Reason'] = FEAT[p]['Reason'].fillna('정상')
    f = REF_FILE.format(p)
    if not os.path.exists(f):                            # 캐시 이름이 다르면 있는 것 중 하나
        cand = sorted(glob.glob(f'./base/ref_{p}_features*.csv'))
        print(p, '지정한 reference 없음 → 후보:', cand); f = cand[0]
    REF[p] = pd.read_csv(f, parse_dates=['TimeStamp'])
    print(p, 'reference:', f, REF[p].shape)

## A-1. 정상 vs 불량 점수 분포 + 임계값 (메인 모델)

In [ ]:
for p in PRODUCTS:
    tb = plot_score_threshold(OOF, MAIN, p, save=f'{OUT}/threshold_{p}.png')
    display(tb)

## A-2. Precision-Recall (운영점 = pct 임계값)

In [ ]:
for p in PRODUCTS:
    plot_pr(OOF, COMPARE, p, save=f'{OUT}/pr_{p}.png')

## A-3. 검사 곡선 — Run 단위 vs 생산계획일 단위 vs 전체 기간

In [ ]:
for p in PRODUCTS:
    for grp in ['run_id', 'plan_date', None]:
        plot_inspection(OOF, COMPARE, p, group_col=grp, save=f'{OUT}/inspection_{p}_{grp or "all"}.png')

## A-4. Run별 점수 타임라인 (메인, pct≥0.995)

In [ ]:
for p in PRODUCTS:
    plot_score_timeline(OOF, MAIN, p, q=0.995, save=f'{OUT}/timeline_{p}.png')

## B-1. 편차 히트맵 — 불량·오탐 샷이 어떤 변수에서 정상 생산 대비 벗어났나
이상탐지 입력과 같은 척도(reference 중앙값·IQR 기준 z). 탐지된 불량(●)은 진한 칸이 있고, 놓친 불량은 전부 옅으면 → **변수로는 정상과 구분 불가**

In [ ]:
HEAT_COLS = EV_DEV + ST_FREE
for p in PRODUCTS:
    deviation_heatmap(FEAT[p], REF[p], HEAT_COLS, OOF, MAIN, p, q=0.995, save=f'{OUT}/heatmap_{p}.png')

## B-2. 단변량 AUC — 변수 하나만으로 얼마나 가르나 (+95% CI)

In [ ]:
UNI_COLS = EV_DEV + EV_DELTA + ST_FREE
for p in PRODUCTS:
    t = univariate_auc(FEAT[p], UNI_COLS)
    t.to_csv(f'{OUT}/univariate_auc_{p}.csv', index=False, encoding='utf-8-sig')
    plot_univariate_auc(t, p, top=15, save=f'{OUT}/univariate_auc_{p}.png')

## B-3. 변수 묶음 ablation (이상탐지 kNN)
⚠️ 이 표를 보고 묶음을 고르면 선택 과적합 → **가설 근거가 있는 묶음만** 채택하거나 중첩 LORO로 확정

In [ ]:
GROUPS = {
    '이벤트 편차': [c for c in EV_DEV if c.startswith('dev_')],
    '충전이상·최대편차': ['fill_anomaly', 'ev_max_abs_dev'],
    '직전차(delta)': EV_DELTA,
    '온도 상태': [c for c in ST_FREE if c != 'dev_flow_res'],
    '흐름저항 편차': ['dev_flow_res'],
}
for p in PRODUCTS:
    t = group_ablation(FEAT[p], REF[p], GROUPS, kind='knn')
    t.to_csv(f'{OUT}/ablation_{p}.csv', index=False, encoding='utf-8-sig')
    display(t.round(3))
    plot_group_ablation(t, p, save=f'{OUT}/ablation_{p}.png')

## B-4. 통합 logit 계수 안정성 (LORO 12 fold)

In [ ]:
from model_v2 import SUP_FSETS
C, S = logit_coef_stability(FEAT, SUP_FSETS['sup_free'])
S.to_csv(f'{OUT}/logit_coef.csv', encoding='utf-8-sig')
display(S.round(3).head(15))
plot_coef_stability(C, S, save=f'{OUT}/logit_coef.png')

## 해석 가이드
| 그래프 | 좋은 신호 | 나쁜 신호 |
|---|---|---|
| A-1 | 불량 점들이 임계값선 위, 정상은 아래 | 불량 분포가 정상과 겹침 (중앙값 비슷) |
| A-2 | 곡선이 무작위선보다 위, 운영점 Precision 높음 | 곡선이 무작위선에 붙음 |
| A-3 | 곡선이 대각선보다 위, 상한에 가까움 | 대각선과 겹침. 단위별로 결과가 크게 다르면 검사 주기 가정에 민감 |
| B-1 | 탐지된 불량에 진한 칸 | 놓친 불량 행이 전부 옅음 → 데이터 한계 |
| B-2 | CI가 0.5를 안 걸침 (채운 점) | 전부 빈 점 → 단일 변수 신호 없음 |
| B-3 | 묶음을 빼면 AP↓ | 빼도 그대로/오히려↑ → 노이즈 묶음 |
| B-4 | 부호 일치율 100%, 0에서 멀리 | 부호가 fold마다 바뀜 → 우연 |